# 02 · LLM financial-mismanagement score from audit reports

For each NHS trust audit report (converted to text):
1. Preprocess the text and extract regex/spaCy features (amounts, risk, governance and negative terms, audit opinion, …)
2. Split it into overlapping chunks
3. Ask the LLM (Gemini 2.0 Flash-Lite) which chunks are most relevant to financial mismanagement
4. Summarise the selected chunks, then combine them into an executive summary
5. Ask the LLM for a **financial mismanagement score (0–1)** from the summary plus the extracted features

**Setup:** `python -m spacy download en_core_web_sm` and set the `GEMINI_API_KEY` environment variable.

In [ ]:
import os
import re
import spacy
from google import genai
import json
import pandas as pd

def preprocess(text):
    """
    Lowercase, remove non-ASCII characters, normalize whitespace.
    """
    text = text.lower()
    # remove non-ASCII
    text = re.sub(r'[^\x00-\x7f]', ' ', text)
    # normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load spaCy model (run python -m spacy download en_core_web_sm first if needed)
nlp = spacy.load("en_core_web_sm")

def extract_financial_features(text):
    """
    Extract financial features from a single NHS trust audit report.
    
    Args:
        text (str): The audit report text to analyze
        
    Returns:
        dict: A dictionary of extracted features with normalized values
    """
    doc = nlp(text)
    features = {}
    
    # Helper function to normalize amounts to millions
    def normalize_amount(amount_str):
        if not amount_str:
            return 0.0
        clean_str = re.sub(r'[^\d.]', '', amount_str)
        if not clean_str:
            return 0.0
        amount = float(clean_str)
        if 'thousand' in amount_str.lower():
            return amount / 1000
        elif 'million' in amount_str.lower():
            return amount
        elif 'billion' in amount_str.lower() or 'bn' in amount_str.lower():
            return amount * 1000
        else:
            return amount / 1e6
    
    # 1. Financial amounts
    amounts = re.findall(r'£\s*(\d{1,3}(?:,\d{3})(?:\.\d{1,2})?|\d+\s(?:million|billion|bn|m|thousand)\b)', text, re.IGNORECASE)
    normalized_amounts = [normalize_amount(amt) for amt in amounts]
    features['total_amount_mentioned'] = sum(normalized_amounts)
    features['num_amount_mentions'] = len(normalized_amounts)
    
    # 2. Context-specific amounts
    context_patterns = {
        'deficit_amount': r'(?:deficit|shortfall)\D{1,20}£\s*([\d,\.]+)',
        'savings_amount': r'(?:savings|efficiency)\D{1,20}£\s*([\d,\.]+)',
        'overspend_amount': r'(?:overspend|exceed)\D{1,20}£\s*([\d,\.]+)'
    }
    for name, pattern in context_patterns.items():
        matches = re.findall(pattern, text, re.IGNORECASE)
        features[name] = normalize_amount(matches[0]) if matches else 0.0
    
    # 3. Risk indicators
    risk_terms = ['risk', 'concern', 'challenge', 'weakness', 'vulnerability']
    features['risk_count'] = sum(1 for token in doc if token.lemma_.lower() in risk_terms)
    
    # 4. Governance indicators
    governance_terms = ['governance', 'control', 'oversight', 'compliance']
    features['governance_count'] = sum(1 for token in doc if token.lemma_.lower() in governance_terms)
    
    # 5. Performance metrics
    features['kpi_mentions'] = len(re.findall(r'\b(?:performance\s+)?(target|metric|indicator|kpi|benchmark)\b', text, re.IGNORECASE))
    
    # 6. Negative sentiment
    negative_terms = ['fail', 'inadequate', 'poor', 'weak', 'insufficient']
    features['negative_count'] = sum(1 for token in doc if token.lemma_.lower() in negative_terms)
    
    # 7. Audit opinion
    opinion_match = re.search(r'\b(qualified\s+opinion|adverse\s+opinion|disclaimer|emphasis\s+of\s+matter)\b', text, re.IGNORECASE)
    features['audit_opinion'] = opinion_match.group(0) if opinion_match else 'unqualified'
    
    # 8. Sentence-level analysis
    deficit_sents = [sent.text for sent in doc.sents if any(t.lemma_.lower() in {'deficit', 'shortfall'} for t in sent)]
    features['deficit_sentences'] = len(deficit_sents)
    
    # 9. Named entities
    money_ents = [ent.text for ent in doc.ents if ent.label_ == 'MONEY']
    features['money_entities'] = len(money_ents)
    
    # 10. Normalized scores
    total_words = len([t for t in doc if t.is_alpha])
    features['risk_density'] = features['risk_count'] / total_words if total_words > 0 else 0
    features['negative_density'] = features['negative_count'] / total_words if total_words > 0 else 0
    
    return features

def preprocess_and_extract_features(text):
    """
    Full pipeline: preprocess text, then extract regex-based features.
    Returns cleaned text and feature dictionary.
    """
    cleaned = preprocess(text)
    features = extract_financial_features(cleaned)
    return cleaned, features


# Quick test on one report
REPORTS_DIR = "../data/reports_text"  # one .txt per trust-year, e.g. R0A2017-2018.txt
filepath = os.path.join(REPORTS_DIR, "R0A2017-2018.txt")
with open(filepath, "r", encoding="utf-8") as f:
    raw_text = f.read()

cleaned_text, features = preprocess_and_extract_features(raw_text)
print("Extracted Features:")
for k, v in features.items():
    print(f"{k}: {v}")


Extracted Features:
total_amount_mentioned: 0
num_amount_mentions: 0
deficit_amount: 0.0
savings_amount: 0.0
overspend_amount: 0.0
risk_count: 245
governance_count: 212
kpi_mentions: 60
negative_count: 6
audit_opinion: unqualified
deficit_sentences: 5
money_entities: 18
risk_density: 0.003613462729712988
negative_density: 8.849296480929766e-05


In [ ]:
def split_overlapping_chunks(text, chunk_size=10000, overlap=100):
    """
    Split `text` into overlapping chunks.

    Args:
        text (str): The input string to split.
        chunk_size (int): The maximum length of each chunk (must be > 0).
        overlap (int): The number of characters by which consecutive chunks overlap (0 <= overlap < chunk_size).

    Returns:
        List[str]: A list of string chunks.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if not (0 <= overlap < chunk_size):
        raise ValueError("overlap must satisfy 0 <= overlap < chunk_size")

    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(text), step):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
    return chunks

def retrieve_top_k_indices(chunks, k, client):
    # build a numbered list of chunks in the prompt
    numbered = "\n\n".join(f"{i+1}:\n{chunks[i]}" for i in range(len(chunks)))
    prompt = (
        f"You are given a query:\n\n  \"Check whether financial mismanagement were conducted.\"\n\n"
        f"And the following {len(chunks)} text chunks, numbered 1–{len(chunks)}:\n\n"
        f"{numbered}\n\n"
        f"Please return the numbers of the {k} chunks most relevant to the query, "
        f"as a comma-separated list (e.g. 3,7,1)."
    )
    resp = client.models.generate_content(
        model="gemini-2.0-flash-lite",
        contents= prompt).text
    # extract ints
    # keep only valid, unique chunk numbers (the LLM can return numbers outside 1..len(chunks))
    indices = []
    for n in re.findall(r"\b(\d+)\b", resp):
        n = int(n)
        if 1 <= n <= len(chunks) and n not in indices:
            indices.append(n)
    return indices[:k]

def summarize_chunk(chunk, client):
    prompt = (
        "Please summarize the following section:\n\n"
        f"{chunk}"
    )
    resp = client.models.generate_content(
        model="gemini-2.0-flash-lite",
        contents= prompt).text    
    return resp.strip()

# --- 3) Synthesize final summary via Gemini ---
def synthesize_summaries(summaries, client):
    combined = "\n\n".join(f"- {s}" for s in summaries)
    prompt = (
        "Here are summaries of the relevant sections:\n\n"
        f"{combined}\n\n"
        "Please produce a cohesive executive summary highlighting the key points."
    )
    resp = client.models.generate_content(
        model="gemini-2.0-flash-lite",
        contents= prompt).text     
    return resp.strip()

def summary_to_score(summary, features, client):
    response = client.models.generate_content(
        model="gemini-2.0-flash-lite",
        contents= f"""
    You are an auditor evaluating the attached summary of an NHS Trust audit report. Also consider some extracted features from the report. 
    Output only a single JSON object with:
    - "financial_mismanagement_score": a numeric value (0.0 to 1.0) for overall likelihood of financial mismanagement or fraud (assume a lower risk if much is redacted/unconfirmed).

    Do not add any commentary or explanation beyond this JSON structure. Do this in a step-by-step manner.
    Here is the report: {summary} and the features {features}.
    """
    ).text 
    
    start_index = response.index('{')
    end_index = response.rindex('}') + 1
    json_content = response[start_index:end_index]

    # 2. Load as JSON.
    data = json.loads(json_content)

    financial_mismanagement_score = data["financial_mismanagement_score"]   
       
    return financial_mismanagement_score   



def text_to_score(text):
    # Set your key first:  export GEMINI_API_KEY="your-key"  (never paste it into the notebook)
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    cleaned_text, features = preprocess_and_extract_features(text)
    chunks = split_overlapping_chunks(cleaned_text)
    top_indices = retrieve_top_k_indices(chunks, k=10, client=client)
    selected = [chunks[i-1] for i in top_indices]
    summaries = [summarize_chunk(c, client=client) for c in selected]
    final_summary = synthesize_summaries(summaries, client=client)
    score = summary_to_score(final_summary, features=features, client=client)
    return score


In [ ]:
df = pd.read_excel("../data/finaldatasetdissertation.xlsx")

folder_path = REPORTS_DIR
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    # Check if it's a file (and not a subfolder)
    if os.path.isfile(file_path):
        print(f"Processing file: {filename}")
        code = filename[0:3]
        year = filename[3:12]
        assert filename[12:] == '.txt', f"Error on file {filename}"
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()    
        financial_mismanagement_score = text_to_score(raw_text)

        # Save score
        mask = (df['Row Labels'] == code) & (df['Year'] == year)
        row_count = mask.sum()  # Number of True values in mask
        assert row_count == 1, f"Expected exactly 1 matching row for (code={code}, year={year}), found {row_count}"
        df.loc[mask, 'financial mismangement score (0-1)'] = financial_mismanagement_score     

In [ ]:
df.to_excel("../data/finaldataset_scored.xlsx", index=False)
print("DataFrame saved")